# Canonical I-JEPA

This notebook validates the **canonical I-JEPA** data pipeline end-to-end (`pretraining.mode=canonical`).  
Standard single-resolution masking (à la original I-JEPA) without cross-scale prediction.

## Goals
- Build `slide_id,wsi_path,mask_path` manifest by pairing files in `data/tcga-prad/wsi` and `data/tcga-prad/tissue-masks`.
- Rebuild slide metadata parquet and anchor catalog manifest with canonical scripts.
- Construct deterministic dataset samples and one collated batch via `make_canonical_loader`.
- Visualize sample images and mask token footprints.
- Run corruption/invariant checks and emit a pass/fail QC summary.

## Outline
1. Setup and imports
2. Config block
3. Manifest construction from local TCGA folders
4. Stage A: slide metadata parquet rebuild
5. Stage B: anchor catalog manifest rebuild
6. Anchor-level corruption guards
7. Stage C: sample-level inspection
8. Visual inspection: sample image
9. Stage D: collated-batch inspection
10. Mask non-leakage and token geometry diagnostics
11. QC summary (fail fast on critical checks)

In [ ]:
from __future__ import annotations

import csv
import json
import random
import subprocess
import sys
from pathlib import Path
from pprint import pprint

import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml

import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists() and (candidate / 'ijepath').exists():
            return candidate
    raise RuntimeError('Could not find repository root containing .git and ijepath package.')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
NOTEBOOK_DIR = REPO_ROOT / 'notebooks'
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from ijepath.datasets.canonical_loader_factory import make_canonical_loader
from ijepath.datasets.canonical_wsi_dataset import CanonicalWSIDataset
from ijepath.datasets.cross_resolution_wsi_dataset import IMAGENET_MEAN, IMAGENET_STD
from ijepath.utils.parquet import require_pyarrow
import canonical_plot_utils

SEED = 21
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f'REPO_ROOT={REPO_ROOT}')
print(f'NOTEBOOK_DIR={NOTEBOOK_DIR}')
print(f'KERNEL_PYTHON={sys.executable}')
print(f'SEED={SEED}')

## Config Block

Fixed defaults used by this notebook.
- `WSI_DIR` / `MASK_DIR`: local slide and tissue-mask folders — both must exist for the manifest step.
- `PROFILE_YAML`: controls anchor geometry (`spacing_tolerance`, FOV sizes, downsample).
- `RUN_YAML`: controls all canonical training hyperparameters (`input_mpp`, tile sizes, mask scales).
- `BATCH_SIZE`: number of samples fetched in Stage D — 2 is enough to exercise collation logic.
- `WSI_BACKEND`: slide reader backend passed to every `WholeSlideImage` call (default: `openslide`).

Canonical params are read from `RUN_YAML`; `spacing_tolerance` from `PROFILE_YAML`.

In [ ]:
EXPECTED_PYTHON = sys.executable

WSI_DIR = Path('../data/tcga-prad/wsi').resolve()
MASK_DIR = Path('../data/tcga-prad/tissue-masks').resolve()
PROFILE_YAML = Path('../configs/profiles/canonical_20x_256_224.yaml').resolve()
RUN_YAML = Path('../configs/runs/canonical_pathorob_camelyon.yaml').resolve()
WORK_DIR = Path('../output/notebook-artifacts/canonical-manifest-to-batch-qc').resolve()

BATCH_SIZE = 8
NUM_WORKERS = 2
WSI_BACKEND = "asap"

MANIFEST_DIR = WORK_DIR / 'manifests'
INDEX_DIR = WORK_DIR / 'indexes'
MPL_DIR = WORK_DIR / 'mpl'
for p in (MANIFEST_DIR, INDEX_DIR, MPL_DIR):
    p.mkdir(parents=True, exist_ok=True)

MANIFEST_CSV = MANIFEST_DIR / 'slides_with_tissue_masks.csv'
SLIDE_METADATA_PARQUET = INDEX_DIR / 'slide_metadata.parquet'
SLIDE_REPORT_CSV = INDEX_DIR / 'slide_metadata_build_report.csv'
ANCHOR_MANIFEST_JSON = INDEX_DIR / 'anchor_catalog_manifest.json'
ANCHOR_REPORT_CSV = INDEX_DIR / 'anchor_catalog_build_report.csv'

QC_RESULTS: list[dict[str, object]] = []

# Read run YAML to extract canonical params
run_cfg = yaml.safe_load(RUN_YAML.read_text(encoding='utf-8'))
canonical_cfg = run_cfg['canonical']

input_mpp = float(canonical_cfg['input_mpp'])
source_tile_size_px = int(canonical_cfg['source_tile_size_px'])
crop_size_px = int(canonical_cfg['crop_size_px'])
enc_mask_scale = tuple(float(x) for x in canonical_cfg['enc_mask_scale'])
pred_mask_scale = tuple(float(x) for x in canonical_cfg['pred_mask_scale'])
aspect_ratio = tuple(float(x) for x in canonical_cfg['aspect_ratio'])
num_enc_masks = int(canonical_cfg['num_enc_masks'])
num_pred_masks = int(canonical_cfg['num_pred_masks'])
min_keep = int(canonical_cfg['min_keep'])

PATCH_SIZE = int(run_cfg['meta']['patch_size'])

# Read profile YAML for downsample and spacing_tolerance
profile_cfg = yaml.safe_load(PROFILE_YAML.read_text(encoding='utf-8'))
downsample = int(profile_cfg.get('downsample', 16))
spacing_tolerance = float(profile_cfg.get('spacing_tolerance', 0.05))


def record_check(name: str, passed: bool, details: str = '') -> None:
    row = {'check': name, 'passed': bool(passed), 'details': str(details)}
    QC_RESULTS.append(row)
    state = 'PASS' if passed else 'FAIL'
    print(f'[{state}] {name}: {details}')
    if not passed:
        raise AssertionError(f'{name} failed: {details}')


def run_cmd(cmd: list[str]) -> str:
    printable = ' '.join(cmd)
    print(f'\n$ {printable}')
    try:
        proc = subprocess.run(
            cmd,
            cwd=str(REPO_ROOT),
            check=True,
            text=True,
            capture_output=True,
        )
    except subprocess.CalledProcessError as exc:
        if exc.stdout:
            print(exc.stdout.strip())
        if exc.stderr:
            print(exc.stderr.strip())
        raise
    if proc.stdout.strip():
        print(proc.stdout.strip())
    if proc.stderr.strip():
        print(proc.stderr.strip())
    return proc.stdout


def tensor_to_rgb_uint8(t: torch.Tensor) -> np.ndarray:
    arr = t.detach().cpu().numpy().transpose(1, 2, 0)
    arr = (arr * IMAGENET_STD[None, None, :]) + IMAGENET_MEAN[None, None, :]
    arr = np.clip(arr, 0.0, 1.0)
    return (arr * 255.0).astype(np.uint8)


for p in (WSI_DIR, MASK_DIR, PROFILE_YAML, RUN_YAML):
    record_check(f'Path exists: {p.name}', p.exists(), str(p))

print(f'input_mpp={input_mpp}, source_tile_size_px={source_tile_size_px}, crop_size_px={crop_size_px}')
print(f'enc_mask_scale={enc_mask_scale}, pred_mask_scale={pred_mask_scale}, aspect_ratio={aspect_ratio}')
print(f'num_enc_masks={num_enc_masks}, num_pred_masks={num_pred_masks}, min_keep={min_keep}')
print(f'downsample={downsample}, spacing_tolerance={spacing_tolerance}')

## Manifest Construction from Directories

Pair `*.tif` files by stem and fail fast on mismatches.

In [ ]:
wsi_by_stem = {p.stem: p.resolve() for p in sorted(WSI_DIR.glob('*.tif'))}
mask_by_stem = {p.stem: p.resolve() for p in sorted(MASK_DIR.glob('*.tif'))}

common_ids = sorted(set(wsi_by_stem) & set(mask_by_stem))
missing_masks = sorted(set(wsi_by_stem) - set(mask_by_stem))
orphan_masks = sorted(set(mask_by_stem) - set(wsi_by_stem))

record_check('At least one matched slide/mask pair', len(common_ids) > 0, f'matched={len(common_ids)}')
record_check('No WSI without mask', len(missing_masks) == 0, str(missing_masks))
record_check('No orphan mask without WSI', len(orphan_masks) == 0, str(orphan_masks))

with MANIFEST_CSV.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['slide_id', 'wsi_path', 'mask_path'])
    writer.writeheader()
    for slide_id in common_ids:
        writer.writerow(
            {
                'slide_id': slide_id,
                'wsi_path': str(wsi_by_stem[slide_id]),
                'mask_path': str(mask_by_stem[slide_id]),
            }
        )

record_check('Manifest CSV written', MANIFEST_CSV.exists(), str(MANIFEST_CSV))

manifest_rows = []
with MANIFEST_CSV.open('r', newline='', encoding='utf-8') as f:
    manifest_rows = list(csv.DictReader(f))

record_check(
    'Manifest row count matches matched ids',
    len(manifest_rows) == len(common_ids),
    f'rows={len(manifest_rows)} matched={len(common_ids)}',
)

print(f'Manifest path: {MANIFEST_CSV}')
display(pd.DataFrame(manifest_rows))

## Stage A: Rebuild Slide Metadata Parquet

The **slide metadata parquet** records each slide's level-0 mpp, available spacings, and mask-to-WSI pixel scale factors.  
Every downstream coordinate conversion (anchor center → pixel region) depends on these values being correct

In [ ]:
run_cmd(
    [
        str(EXPECTED_PYTHON),
        str(REPO_ROOT / 'scripts/build_slide_metadata_index_from_manifest.py'),
        '--manifest', str(MANIFEST_CSV),
        '--output', str(SLIDE_METADATA_PARQUET),
        '--report', str(SLIDE_REPORT_CSV),
        '--backend', WSI_BACKEND,
    ]
)

record_check('Slide metadata parquet exists', SLIDE_METADATA_PARQUET.exists(), str(SLIDE_METADATA_PARQUET))
record_check('Slide metadata report exists', SLIDE_REPORT_CSV.exists(), str(SLIDE_REPORT_CSV))

with SLIDE_REPORT_CSV.open('r', newline='', encoding='utf-8') as f:
    slide_report_rows = list(csv.DictReader(f))
failed_slides = [r for r in slide_report_rows if str(r.get('status')) != 'ok']
record_check('All slide metadata rows succeeded', len(failed_slides) == 0, f'failed={len(failed_slides)}')

_, _, ds = require_pyarrow()
slide_table = ds.dataset(str(SLIDE_METADATA_PARQUET), format='parquet').to_table()
preview_rows = slide_table.slice(0, min(3, slide_table.num_rows)).to_pylist()
display(pd.DataFrame(preview_rows))

## Stage B: Rebuild Anchor Catalog Manifest

The **anchor catalog** defines valid tile centers that satisfy tissue coverage and boundary constraints.

In [ ]:
run_cmd(
    [
        str(EXPECTED_PYTHON),
        str(REPO_ROOT / 'scripts/build_valid_context_anchor_catalog.py'),
        '--slide-index', str(SLIDE_METADATA_PARQUET),
        '--profile', str(PROFILE_YAML),
        '--output', str(ANCHOR_MANIFEST_JSON),
        '--backend', WSI_BACKEND,
        '--workers', '1',
    ]
)

record_check('Anchor manifest JSON exists', ANCHOR_MANIFEST_JSON.exists(), str(ANCHOR_MANIFEST_JSON))
record_check('Anchor build report exists', ANCHOR_REPORT_CSV.exists(), str(ANCHOR_REPORT_CSV))

anchor_manifest = json.loads(ANCHOR_MANIFEST_JSON.read_text(encoding='utf-8'))
anchor_shards = list(anchor_manifest.get('anchor_shards', []))
total_anchors = int(anchor_manifest.get('total_anchors', 0))

record_check('Anchor manifest has non-zero anchors', total_anchors > 0, f'total_anchors={total_anchors}')
record_check('Anchor manifest has shard entries', len(anchor_shards) > 0, f'shards={len(anchor_shards)}')

rows_from_manifest = int(sum(int(s.get('rows', 0)) for s in anchor_shards))
record_check(
    'Shard row sum equals manifest total_anchors',
    rows_from_manifest == total_anchors,
    f'shard_rows={rows_from_manifest} total={total_anchors}',
)

missing_shards = [str(Path(s['path'])) for s in anchor_shards if not Path(s['path']).exists()]
record_check('All shard files exist on disk', len(missing_shards) == 0, str(missing_shards))

with ANCHOR_REPORT_CSV.open('r', newline='', encoding='utf-8') as f:
    anchor_report_rows = list(csv.DictReader(f))
failed_anchor_slides = [r for r in anchor_report_rows if str(r.get('status')) != 'ok']
record_check(
    'All anchor report slide statuses are ok',
    len(failed_anchor_slides) == 0,
    f'failed={len(failed_anchor_slides)}',
)

stratum_counts = anchor_manifest.get('stratum_counts', {})
print('Anchor profile:')
pprint(anchor_manifest.get('profile', {}))
print('Sampling stratum key:', anchor_manifest.get('sampling_stratum_key'))
print('Stratum counts:', stratum_counts)
display(pd.DataFrame(anchor_report_rows))

## Anchor-Level Integrity Checks (Corruption Guards)

Read one shard parquet and verify required fields, numeric finiteness, tissue range, and path existence.

**Tissue fraction** is the proportion of tissue in the anchor: a value of 0.2 means 20% of the sampled region is tissue.  
The anchor catalog enforces a minimum threshold during construction.  
Checking it here guards against catalog regeneration bugs or schema drift.

In [ ]:
import pyarrow.parquet as pq

first_shard_path = Path(anchor_shards[0]['path']).resolve()
anchor_table = pq.read_table(str(first_shard_path))
anchor_rows = anchor_table.to_pylist()
record_check('First anchor shard has rows', anchor_table.num_rows > 0, f'rows={anchor_table.num_rows}')

required_cols = {
    'anchor_id',
    'slide_id',
    'wsi_path',
    'mask_path',
    'center_x_level0',
    'center_y_level0',
    'tissue_fraction',
    'context_mpp',
    'target_mpp',
    'context_fov_um',
    'target_fov_um',
    'targets_per_context',
}
missing_cols = sorted(required_cols - set(anchor_table.column_names))
record_check('Required anchor columns present', len(missing_cols) == 0, str(missing_cols))

tissue = np.asarray(anchor_table.column('tissue_fraction').to_numpy(), dtype=np.float64)
record_check('All tissue fractions finite', bool(np.isfinite(tissue).all()), '')
record_check(
    'All tissue fractions in [0,1]',
    bool(((tissue >= 0.0) & (tissue <= 1.0)).all()),
    f'min={float(tissue.min()):.5f} max={float(tissue.max()):.5f}',
)

numeric_cols = [
    'center_x_level0',
    'center_y_level0',
    'context_mpp',
    'target_mpp',
    'context_fov_um',
    'target_fov_um',
    'targets_per_context',
]
for col in numeric_cols:
    vals = np.asarray(anchor_table.column(col).to_numpy(), dtype=np.float64)
    record_check(f'Column finite: {col}', bool(np.isfinite(vals).all()), col)

unique_wsi_paths = sorted({str(r['wsi_path']) for r in anchor_rows})
unique_mask_paths = sorted({str(r['mask_path']) for r in anchor_rows})
missing_wsi_paths = [p for p in unique_wsi_paths if not Path(p).exists()]
missing_mask_paths = [p for p in unique_mask_paths if not Path(p).exists()]
record_check('All anchor shard WSI paths exist', len(missing_wsi_paths) == 0, str(missing_wsi_paths))
record_check('All anchor shard mask paths exist', len(missing_mask_paths) == 0, str(missing_mask_paths))

display(pd.DataFrame(anchor_rows[:5]))

In [ ]:
# per-slide lookups and anchor row index (shared by Stage C and Stage D)
from canonical_qc_utils import (
    load_source_tile_and_mask,
    build_slide_lookups,
    load_anchor_index,
)

_slide_meta_rows = ds.dataset(str(SLIDE_METADATA_PARQUET), format='parquet').to_table().to_pylist()
(
    wsi_l0_mpp_by_slide,
    mask_spacings_by_slide,
    mask_scale_x_by_slide,
    mask_scale_y_by_slide
) = build_slide_lookups(_slide_meta_rows)

anchor_data_by_id = load_anchor_index(anchor_shards)

print(f'Loaded {len(anchor_data_by_id)} anchor rows | {len(wsi_l0_mpp_by_slide)} slides')

## Stage C: Dataset Sample Inspection

Instantiate `CanonicalWSIDataset` directly; fetch deterministic samples and verify shape, finiteness, and metadata keys.

**Determinism mechanism:** PyTorch's global RNG is seeded with `SEED + index` before each `__getitem__` call.  
This makes `RandomResizedCrop` deterministic for a given index, so the same sample always produces the same crop.  
Essential for reproducible visual inspection and regression testing.

In [ ]:
sample_dataset = CanonicalWSIDataset(
    anchor_catalog_manifest=str(ANCHOR_MANIFEST_JSON),
    input_mpp=input_mpp,
    source_tile_size_px=source_tile_size_px,
    crop_size_px=crop_size_px,
    patch_size=PATCH_SIZE,
    seed=SEED,
    spacing_tolerance=spacing_tolerance,
    backend=WSI_BACKEND,
)

print('len(dataset):', len(sample_dataset))
print('crop_size_px:', sample_dataset.crop_size_px)
print('source_tile_size_px:', sample_dataset.source_tile_size_px)

# Seed per-index so RandomResizedCrop is deterministic for the same index
torch.manual_seed(SEED + 0)
sample0 = sample_dataset[0]
torch.manual_seed(SEED + 1)
sample1 = sample_dataset[1]
torch.manual_seed(SEED + 0)
sample0_repeat = sample_dataset[0]

record_check(
    'Deterministic sample[0] image tensor',
    bool(torch.allclose(sample0['image'], sample0_repeat['image'])),
    'sample0 vs sample0_repeat',
)

for idx, sample in enumerate([sample0, sample1]):
    img = sample['image']
    m = sample['sample_metadata']

    record_check(
        f'sample[{idx}] image shape',
        img.ndim == 3 and tuple(img.shape) == (3, crop_size_px, crop_size_px),
        str(tuple(img.shape)),
    )
    record_check(f'sample[{idx}] image finite', bool(torch.isfinite(img).all().item()), '')

    required_meta = {
        'slide_id',
        'anchor_id',
        'requested_input_mpp',
        'source_input_mpp',
        'source_resolution_mode',
        'model_crop_size_px',
    }
    missing_meta = sorted(required_meta - set(m.keys()))
    record_check(f'sample[{idx}] required metadata keys', len(missing_meta) == 0, str(missing_meta))

print('sample0 metadata keys:', sorted(sample0['sample_metadata'].keys()))

## Visual Inspection: Sample Image

Denormalize and display `sample0['image']`; print metadata table.

In [ ]:
sample_viz_path = MPL_DIR / 'sample0_canonical.png'
fig = canonical_plot_utils.visualize_sample(
    sample=sample0,
    anchor_data_by_id=anchor_data_by_id,
    wsi_l0_mpp_by_slide=wsi_l0_mpp_by_slide,
    source_tile_size_px=source_tile_size_px,
    input_mpp=input_mpp,
    spacing_tolerance=spacing_tolerance,
    downsample=downsample,
    backend=WSI_BACKEND,
    load_fn=load_source_tile_and_mask,
    record_check_fn=record_check,
)
fig.savefig(sample_viz_path, dpi=170, bbox_inches='tight')
plt.show()
record_check('Sample visualization written', sample_viz_path.exists(), str(sample_viz_path))

## Stage D: Collated Batch Inspection

Build loader via `make_canonical_loader`, fetch one batch, and verify tensor shapes and mask counts.

**Determinism mechanism:** the loader uses a `seed`-initialized sampler; with `NUM_WORKERS=0` and a fixed `SEED`, the first batch is reproducible across runs. Worker-based randomness (augmentation) is seeded per-index by the dataset, so batch content is stable for inspection.

In [ ]:
loader_dataset, loader, _ = make_canonical_loader(
    batch_size=BATCH_SIZE,
    pin_mem=False,
    num_workers=NUM_WORKERS,
    world_size=1,
    rank=0,
    drop_last=False,
    anchor_catalog_manifest=str(ANCHOR_MANIFEST_JSON),
    patch_size=PATCH_SIZE,
    input_mpp=input_mpp,
    source_tile_size_px=source_tile_size_px,
    crop_size_px=crop_size_px,
    crop_scale=(0.3, 1.0),
    use_horizontal_flip=True,
    horizontal_flip_prob=0.5,
    use_color_distortion=False,
    color_jitter_strength=0.0,
    use_gaussian_blur=False,
    enc_mask_scale=enc_mask_scale,
    pred_mask_scale=pred_mask_scale,
    aspect_ratio=aspect_ratio,
    num_enc_masks=num_enc_masks,
    num_pred_masks=num_pred_masks,
    min_keep=min_keep,
    allow_overlap=False,
    seed=SEED,
    spacing_tolerance=spacing_tolerance,
    backend=WSI_BACKEND,
    sampling_strategy='stratified_weighted',
    sampling_stratum_key='organ',
    sampling_stratum_weights='inverse_frequency',
    persistent_workers=False,
    prefetch_factor=2,
    max_open_slides_per_worker=8,
    anchor_stream_batch_size=256,
)

batch_data, masks_enc, masks_pred = next(iter(loader))

print('image:', tuple(batch_data['image'].shape))
print('num encoder masks:', len(masks_enc), 'shape[0]:', tuple(masks_enc[0].shape))
print('num predictor masks:', len(masks_pred), 'shape[0]:', tuple(masks_pred[0].shape))

record_check(
    'Batch image shape',
    tuple(batch_data['image'].shape) == (BATCH_SIZE, 3, crop_size_px, crop_size_px),
    str(tuple(batch_data['image'].shape)),
)
record_check('Batch image finite', bool(torch.isfinite(batch_data['image']).all().item()), '')
record_check(
    'Encoder mask count matches num_enc_masks',
    len(masks_enc) == num_enc_masks,
    f'len={len(masks_enc)} expected={num_enc_masks}',
)
record_check(
    'Predictor mask count matches num_pred_masks',
    len(masks_pred) == num_pred_masks,
    f'len={len(masks_pred)} expected={num_pred_masks}',
)

for i, mask in enumerate(masks_enc):
    record_check(f'masks_enc[{i}] rank 2', mask.ndim == 2, str(tuple(mask.shape)))
    record_check(f'masks_enc[{i}] batch size', mask.shape[0] == BATCH_SIZE, str(tuple(mask.shape)))
    record_check(f'masks_enc[{i}] non-empty', mask.shape[1] > 0, str(tuple(mask.shape)))

for i, mask in enumerate(masks_pred):
    record_check(f'masks_pred[{i}] rank 2', mask.ndim == 2, str(tuple(mask.shape)))
    record_check(f'masks_pred[{i}] batch size', mask.shape[0] == BATCH_SIZE, str(tuple(mask.shape)))
    record_check(f'masks_pred[{i}] non-empty', mask.shape[1] > 0, str(tuple(mask.shape)))

### Batch Visualization

Each row is one sample in the batch.  
Columns: raw anchor | augmented crop | context tokens | one panel per target  

In canonical I-JEPA, context token count is controlled by the canonical mask collator (defined in `multiblock.py`).  
The logic for context tokens kept is influenced by 5 steps:

1. Total token grid is fixed by crop_size_px / patch_size<br>
N = (crop_size_px / patch_size)^2<br>
for 224 crops and patch 16, **N=196**

2. Context mask size is sampled from `canonical.enc_mask_scale` (which is set to (0.85, 1.0) here)
Expected keep count is roughly enc_mask_scale * N

3. Context mask placement is constrained by target masks when `allow_overlap=false`
So effective keep tokens can shrink depending on sampled target blocks and overlap constraints.

4. `min_keep` is a floor for valid sampled context masks
Masks smaller than this are rejected/resampled.

5. Batch-level truncation makes all samples same length
After sampling, it truncates every context mask to the batch minimum (`min_keep_enc`) so tensors stack cleanly.

In [ ]:
batch_viz_path = MPL_DIR / 'batch_overview.png'
fig = canonical_plot_utils.visualize_batch(
    batch_data=batch_data,
    masks_enc=masks_enc,
    masks_pred=masks_pred,
    anchor_data_by_id=anchor_data_by_id,
    wsi_l0_mpp_by_slide=wsi_l0_mpp_by_slide,
    source_tile_size_px=source_tile_size_px,
    input_mpp=input_mpp,
    patch_size=PATCH_SIZE,
    downsample=downsample,
    spacing_tolerance=spacing_tolerance,
    backend=WSI_BACKEND,
    load_fn=load_source_tile_and_mask,
    record_check_fn=record_check,
)
fig.savefig(batch_viz_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {batch_viz_path}')

## Mask Non-Leakage and Token Geometry Checks

Checks:
- `encoder_keep_tokens ∩ predictor_union_tokens == ∅` (non-leakage invariant)
- Token counts are in expected ranges
- Visual overlay of encoder-keep and predictor masks on the sample image

**Why non-leakage is critical:** in I-JEPA, the context encoder must never see the target tokens.  
If it does, the predictor can trivially copy features rather than learning to predict them from context.  
A leaking mask collapses training: the model gets the answer for free and learns nothing useful about spatial structure.

In [ ]:
sample_idx = 0
grid_h = crop_size_px // PATCH_SIZE
grid_w = crop_size_px // PATCH_SIZE
total_tokens = grid_h * grid_w

print(f'Token grid: {grid_h}x{grid_w} = {total_tokens} tokens')

enc_tokens = set(masks_enc[0][sample_idx].detach().cpu().tolist())
pred_token_sets = [set(pm[sample_idx].detach().cpu().tolist()) for pm in masks_pred]
pred_union = set().union(*pred_token_sets)
leak_overlap = sorted(enc_tokens & pred_union)

record_check(
    'Non-leakage: encoder keep ∩ predictor union == empty',
    len(leak_overlap) == 0,
    f'overlap_tokens={len(leak_overlap)}',
)

print(f'Encoder kept: {len(enc_tokens)}/{total_tokens} tokens')
print(f'Predictor union: {len(pred_union)}/{total_tokens} tokens')
for i, ps in enumerate(pred_token_sets):
    print(f'  pred mask {i + 1}: {len(ps)} tokens')

# Visualize encoder-keep and predictor masks overlaid on the sample image
img_rgb0 = tensor_to_rgb_uint8(batch_data['image'][sample_idx])
fig = canonical_plot_utils.plot_canonical_masks(
    image_rgb=img_rgb0,
    masks_enc=masks_enc,
    masks_pred=masks_pred,
    grid_h=grid_h,
    grid_w=grid_w,
    sample_idx=sample_idx,
)
mask_viz_path = MPL_DIR / 'mask_geometry_sample0.png'
fig.savefig(mask_viz_path, dpi=170, bbox_inches='tight')
plt.show()

record_check('Mask visualization written', mask_viz_path.exists(), str(mask_viz_path))

## QC Summary

Consolidate all checks and fail if any critical check did not pass.

**What to do if something fails:**
- *Path exists* failures → check `WSI_DIR`, `MASK_DIR`, and config YAML paths in the Config Block.
- *Stage A/B script failures* → inspect the `_build_report.csv` files for per-slide errors; common causes are corrupted slide files or missing openslide drivers.
- *Anchor shard* failures → re-run Stage B; shard paths are absolute and may be stale after a directory move.
- *Shape / finiteness failures* → check `input_mpp` and `source_tile_size_px` in `RUN_YAML` against what the slides support.
- *Non-leakage failure* → this is a code bug in the mask collator; do not proceed to training until resolved.

In [ ]:
failed = [r for r in QC_RESULTS if not bool(r['passed'])]
assert not failed, f'QC failed with {len(failed)} failing checks: {failed}'

print(f'\nAll critical QC checks passed: {len(QC_RESULTS)} checks')
print('Artifacts:')
print(f'- Manifest CSV: {MANIFEST_CSV}')
print(f'- Slide metadata parquet: {SLIDE_METADATA_PARQUET}')
print(f'- Anchor manifest: {ANCHOR_MANIFEST_JSON}')
print(f'- Sample image: {sample_viz_path}')
print(f'- Mask visualization: {mask_viz_path}')

qc_df = pd.DataFrame(QC_RESULTS)
display(qc_df)